# Tutorial 4: Classification for Credit Risk

**Big Data in Finance** | Dr Daniele Bianchi  
**Duration:** 1 hour

---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Build classification models (Logistic Regression, Random Forest, Gradient Boosting) for credit risk
2. Evaluate models using the **confusion matrix**, **precision**, **recall**, **F1**, and **AUC**
3. Choose the **optimal threshold** based on business costs
4. Translate model predictions into **business decisions**

---

## Business Context

You work at a peer-to-peer lending platform (LendingClub). For each loan application, you need to:
- **Predict** whether the borrower will default
- **Decide** whether to approve or deny the loan
- **Set** an appropriate interest rate based on risk

The costs of errors are **asymmetric**:
- Denying a good borrower → Lost profit (~£500)
- Approving a defaulter → Lost principal (~£5,000)

> 💡 **Note (scope):** This tutorial uses a deliberately simplified setup for a one-hour session, so its numbers will not exactly reproduce the full results reported in the lecture slides and notes.

---

## Part 1: Setup and Data Preparation

> ⚠️ **Leakage caveat:** This dataset includes `int_rate` and `grade`, which are LendingClub's *own* risk-based price and rating — they already encode LendingClub's default expectation. Using them as features makes the task easier than a genuine origination-time model would be. We keep them here for simplicity; in practice you would drop them (or model them separately) to avoid this subtle leakage.

In [ ]:
# =============================================================================
# Import libraries
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation metrics
from sklearn.metrics import (
    confusion_matrix,           # Counts of TP, TN, FP, FN
    classification_report,      # Precision, recall, F1 summary
    roc_curve, auc,            # ROC curve and AUC
    roc_auc_score,             # AUC score directly
    ConfusionMatrixDisplay     # Visualization
)

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Libraries loaded!")

In [ ]:
# =============================================================================
# Load the LendingClub dataset
# =============================================================================

# This is real peer-to-peer lending data from LendingClub
# Each row is a loan application with borrower characteristics

df = pd.read_csv('lending_club_tutorial.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns:")
print(df.columns.tolist())

In [ ]:
# =============================================================================
# Explore the data
# =============================================================================

# Look at the first few rows
print("First 5 rows:")
df.head()

In [ ]:
# =============================================================================
# Understand the target variable
# =============================================================================

# 'default' is our target: 1 = borrower defaulted, 0 = loan repaid

print("Target variable distribution:")
print(df['default'].value_counts())
print(f"\nDefault rate: {df['default'].mean():.1%}")
print(f"  - Non-defaults: {(df['default']==0).sum():,}")
print(f"  - Defaults: {(df['default']==1).sum():,}")

In [ ]:
# =============================================================================
# Understand the features
# =============================================================================

# Feature descriptions:
# loan_amnt: The listed amount of the loan applied for
# int_rate: Interest rate on the loan
# installment: Monthly payment amount
# grade: LC-assigned loan grade (categorical: A, B, C, D, E, F, G)
# annual_inc: Annual income of the borrower
# home_ownership: Home ownership status (categorical)
# verification_status: Whether income was verified (categorical)
# purpose: Purpose of the loan (categorical)
# addr_state: Borrower's state of residence (categorical)
# dti: Debt-to-income ratio
# open_acc: Number of open credit lines
# total_acc: Total number of credit lines
# revol_bal: Total revolving balance
# revol_util: Revolving utilisation rate (%)
# pub_rec: Number of derogatory public records
# inq_last_6mths: Number of credit inquiries in the last 6 months
# delinq_2yrs: Number of 30+ day delinquencies in the last 2 years
# emp_length: Employment length in years
# term: Loan term in months
# credit_history: Length of credit history

print("Feature summary statistics:")
df.describe().round(2)

In [ ]:
# =============================================================================
# Handle categorical variables
# =============================================================================

# Identify categorical columns that need encoding
categorical_cols = ['grade', 'home_ownership', 'verification_status', 'purpose','addr_state']

# Check which categorical columns exist in our data
existing_cat_cols = [col for col in categorical_cols if col in df.columns]
print(f"Categorical columns found: {existing_cat_cols}")

# One-hot encode categorical variables
# drop_first=True avoids multicollinearity (dummy variable trap)
if existing_cat_cols:
    df_encoded = pd.get_dummies(df, columns=existing_cat_cols, drop_first=True)
else:
    df_encoded = df.copy()

print(f"\nShape after encoding: {df_encoded.shape}")

In [ ]:
# =============================================================================
# Handle missing values
# =============================================================================

# Check for missing values (after one-hot encoding, only numeric columns can have NaNs)
missing = df_encoded.isnull().sum()
missing_cols = missing[missing > 0]

if len(missing_cols) > 0:
    print("Columns with missing values:")
    print(missing_cols)
else:
    print("No missing values found!")

# NOTE: we impute AFTER the train/test split, using training-set medians only,
# to avoid leaking test-set information into the imputation (see the split cell below).

In [ ]:
# =============================================================================
# Prepare features and target
# =============================================================================

# Target variable
y = df_encoded['default']

# Features: everything except the target
X = df_encoded.drop('default', axis=1)

# Store feature names for later
feature_names = X.columns.tolist()

print(f"Number of features: {len(feature_names)}")
print(f"Features: {feature_names[:10]}...")  # Show first 10

In [ ]:
# =============================================================================
# Split into train and test sets, then impute (training statistics only)
# =============================================================================

# stratify=y ensures the same default rate in both train and test sets
# This is important for imbalanced data!

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,      # 30% for testing
    random_state=42,
    stratify=y          # Maintain class balance in both sets
)

# Impute missing values using TRAINING medians only (prevents leakage from the test set)
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"\nDefault rate in training: {y_train.mean():.1%}")
print(f"Default rate in test: {y_test.mean():.1%}")
print(f"Remaining missing values: train={int(X_train.isna().sum().sum())}, test={int(X_test.isna().sum().sum())}")

In [ ]:
# =============================================================================
# Scale features (required for Logistic Regression)
# =============================================================================

# Logistic Regression with regularization needs scaled features
# Tree-based methods (RF, GB) don't need scaling, but it doesn't hurt

scaler = StandardScaler()

# IMPORTANT: Fit on training data only, then transform both
# This prevents data leakage from test set
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled!")

---

## Part 2: Train Classification Models

We'll train three models and compare them:
1. **Logistic Regression** — interpretable baseline
2. **Random Forest** — ensemble of trees
3. **Gradient Boosting** — typically best accuracy

In [ ]:
# =============================================================================
# 1. Logistic Regression
# =============================================================================

# Logistic regression models the probability of default:
# P(default | X) = 1 / (1 + exp(-X'β))
#
# class_weight='balanced' automatically adjusts for imbalanced data:
# - Gives more weight to the minority class (defaults)
# - Prevents the model from just predicting "no default" for everything

logreg = LogisticRegression(
    penalty='l2',               # L2 regularization (Ridge-style)
    C=1.0,                      # Inverse regularization strength
    class_weight='balanced',    # Handle class imbalance
    max_iter=1000,              # Ensure convergence
    random_state=42
)

# Fit on scaled training data
logreg.fit(X_train_scaled, y_train)

print("Logistic Regression fitted!")

In [ ]:
# =============================================================================
# 2. Random Forest Classifier
# =============================================================================

# Random Forest: Ensemble of decision trees
# - Each tree is trained on a bootstrap sample
# - Each tree considers a random subset of features
# - Final prediction = majority vote (or average probability)

rf = RandomForestClassifier(
    n_estimators=200,          # Number of trees
    max_depth=10,              # Limit tree depth to prevent overfitting
    min_samples_leaf=20,       # Minimum samples per leaf
    class_weight='balanced',   # Handle class imbalance
    random_state=42,
    n_jobs=-1                  # Use all CPU cores
)

# Fit on unscaled data (trees don't need scaling)
rf.fit(X_train, y_train)

print("Random Forest fitted!")

In [ ]:
# =============================================================================
# 3. Gradient Boosting Classifier
# =============================================================================

# Gradient Boosting: Trees built sequentially
# - Each tree corrects the errors of previous trees
# - Usually achieves best accuracy with proper tuning
# - More prone to overfitting than Random Forest
# - Be careful, with n_estimators=100 the estimation can take some time

gb = GradientBoostingClassifier(
    n_estimators=100,          # Number of boosting rounds
    max_depth=4,               # Keep trees shallow
    learning_rate=0.01,        # Learning rate (or shrinkage factor)
    subsample=0.8,             # Stochastic gradient boosting
    random_state=42
)

# Fit the model
gb.fit(X_train, y_train)

print("Gradient Boosting fitted!")

In [ ]:
# =============================================================================
# Generate predictions and probabilities for all models
# =============================================================================

# For each model, we get:
# 1. predict(): Class labels (0 or 1) using default threshold of 0.5
# 2. predict_proba(): Probabilities for each class [P(0), P(1)]

# Logistic Regression (uses scaled data)
y_pred_logreg = logreg.predict(X_test_scaled)
y_prob_logreg = logreg.predict_proba(X_test_scaled)[:, 1]  # P(default)

# Random Forest (uses unscaled data)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Gradient Boosting (uses unscaled data)
y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

print("Predictions generated!")
print(f"\nExample probabilities (first 5 test samples):")
print(f"  Logistic: {y_prob_logreg[:5].round(3)}")
print(f"  RF:       {y_prob_rf[:5].round(3)}")
print(f"  GB:       {y_prob_gb[:5].round(3)}")

---

## Part 3: The Confusion Matrix

The confusion matrix shows the counts of correct and incorrect predictions.

In [ ]:
# =============================================================================
# Display confusion matrices for all models
# =============================================================================

# Confusion matrix layout:
#                 Predicted
#              No Default | Default
# Actual No Default:  TN  |  FP
# Actual Default:     FN  |  TP
#
# TN = True Negative: Correctly predicted no default
# FP = False Positive: Predicted default, but didn't (denied good borrower)
# FN = False Negative: Predicted no default, but did (approved bad borrower) ← COSTLY!
# TP = True Positive: Correctly predicted default

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, y_pred) in zip(axes, [
    ('Logistic Regression', y_pred_logreg),
    ('Random Forest', y_pred_rf),
    ('Gradient Boosting', y_pred_gb)
]):
    # Create confusion matrix display
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['No Default', 'Default'],
        cmap='Blues',
        ax=ax
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Extract confusion matrix values for Logistic Regression
# =============================================================================

# Get the raw confusion matrix
cm = confusion_matrix(y_test, y_pred_logreg)

# Extract individual values
TN, FP, FN, TP = cm.ravel()

print("Logistic Regression Confusion Matrix:")
print(f"  True Negatives (TN):  {TN:,} - Correctly approved good borrowers")
print(f"  False Positives (FP): {FP:,} - Wrongly denied good borrowers")
print(f"  False Negatives (FN): {FN:,} - Wrongly approved defaulters ⚠️")
print(f"  True Positives (TP):  {TP:,} - Correctly denied defaulters")

---

## Part 4: Evaluation Metrics

### 4.1 Precision, Recall, and F1 Score

In [ ]:
# =============================================================================
# Calculate and understand key metrics
# =============================================================================

# PRECISION: Of all loans we flagged as "default", what % actually defaulted?
# Precision = TP / (TP + FP)
# Interpretation: How reliable are our "deny" decisions?
precision = TP / (TP + FP) if (TP + FP) > 0 else 0

# RECALL (Sensitivity): Of all actual defaults, what % did we catch?
# Recall = TP / (TP + FN)  
# Interpretation: How many defaults did we miss?
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# F1 SCORE: Harmonic mean of precision and recall
# F1 = 2 * (Precision * Recall) / (Precision + Recall)
# Interpretation: Single metric balancing both
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# ACCURACY: Overall correct predictions (can be misleading!)
accuracy = (TP + TN) / (TP + TN + FP + FN)

print("Logistic Regression Metrics (at threshold = 0.5):")
print("="*50)
print(f"Precision: {precision:.3f} ({precision*100:.1f}%)")
print(f"  → Of loans flagged as risky, {precision*100:.1f}% actually defaulted")
print(f"")
print(f"Recall:    {recall:.3f} ({recall*100:.1f}%)")
print(f"  → We caught {recall*100:.1f}% of all defaults")
print(f"")
print(f"F1 Score:  {f1:.3f}")
print(f"")
print(f"Accuracy:  {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"  ⚠️ Accuracy can be misleading with imbalanced data!")

In [ ]:
# =============================================================================
# Compare all models using classification_report
# =============================================================================

print("Model Comparison (at threshold = 0.5):")
print("="*60)

for name, y_pred in [
    ('Logistic Regression', y_pred_logreg),
    ('Random Forest', y_pred_rf),
    ('Gradient Boosting', y_pred_gb)
]:
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

### 4.2 ROC Curve and AUC

In [ ]:
# =============================================================================
# Plot ROC curves for all models
# =============================================================================

# ROC curve shows the trade-off between:
# - True Positive Rate (TPR) = Recall = TP / (TP + FN)
# - False Positive Rate (FPR) = FP / (FP + TN)
#
# As we lower the threshold:
# - We predict "default" more often
# - TPR increases (catch more defaults)
# - FPR also increases (more false alarms)

plt.figure(figsize=(6, 6))

# Calculate and plot ROC curve for each model
for name, y_prob, color in [
    ('Logistic Regression', y_prob_logreg, 'blue'),
    ('Random Forest', y_prob_rf, 'green'),
    ('Gradient Boosting', y_prob_gb, 'red')
]:
    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    # Plot
    plt.plot(fpr, tpr, color=color, linewidth=2, 
             label=f'{name} (AUC = {roc_auc:.3f})')

# Plot diagonal (random classifier)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR / Recall)')
plt.title('ROC Curves: Model Comparison')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 AUC Interpretation:")
print("   AUC = probability that model ranks a random defaulter higher than a random non-defaulter")
print("   Higher AUC = better ranking ability across all thresholds")

### 🤖 AI Prompt Exercise 1: Interpret the Metrics

**Task:** Ask an AI to help interpret your evaluation results.

**Copy this prompt (fill in your values):**

```
I built a credit risk model with these results at threshold=0.5:

- Precision: [YOUR VALUE]
- Recall: [YOUR VALUE]
- AUC: [YOUR VALUE]

Questions:
1. If precision is 35%, what does that mean for my business decisions?
2. If recall is 65%, what percentage of defaults am I missing?
3. Is my AUC score good, average, or poor for credit risk?

Explain in business terms (4-5 sentences).
```

**Write the key insights:**

*Your notes:*



---

## Part 5: Threshold Optimization

The default threshold of 0.5 is **rarely optimal** for business decisions!

### 5.1 Why 0.5 is Wrong

In [ ]:
# =============================================================================
# The cost of different errors
# =============================================================================

# In credit risk, errors have ASYMMETRIC costs:
#
# False Positive (FP): Deny a good borrower
#   Cost: Lost profit on the loan (maybe £500)
#
# False Negative (FN): Approve a defaulter  
#   Cost: Lost principal (maybe £5,000)
#
# Defaults are 10x more costly!

COST_FP = 500    # Cost of denying a good borrower (lost profit)
COST_FN = 5000   # Cost of approving a defaulter (lost principal)

print("Cost Structure:")
print(f"  False Positive (deny good borrower): £{COST_FP:,}")
print(f"  False Negative (approve defaulter):  £{COST_FN:,}")
print(f"\n  Ratio: Defaults are {COST_FN/COST_FP:.0f}x more costly!")

In [ ]:
# =============================================================================
# Calculate total cost at different thresholds
# =============================================================================

def calculate_cost_at_threshold(y_true, y_prob, threshold, cost_fp, cost_fn):
    """
    Calculate total cost of errors at a given threshold.
    
    Parameters:
    -----------
    y_true : array, actual labels (0 or 1)
    y_prob : array, predicted probabilities
    threshold : float, classification threshold
    cost_fp : float, cost of false positive
    cost_fn : float, cost of false negative
    
    Returns:
    --------
    dict with FP count, FN count, total cost, and metrics
    """
    # Make predictions at this threshold
    y_pred = (y_prob >= threshold).astype(int)
    
    # Get confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    
    # Calculate total cost
    total_cost = (FP * cost_fp) + (FN * cost_fn)
    
    # Calculate metrics
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    
    return {
        'threshold': threshold,
        'FP': FP,
        'FN': FN,
        'total_cost': total_cost,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# =============================================================================
# Find the optimal threshold by testing many values
# =============================================================================

# Test thresholds from 0.05 to 0.95
thresholds = np.arange(0.05, 0.96, 0.01)

# Calculate cost at each threshold for Gradient Boosting (typically best model)
results = []
for t in thresholds:
    result = calculate_cost_at_threshold(y_test, y_prob_gb, t, COST_FP, COST_FN)
    results.append(result)

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Find optimal threshold (minimum total cost)
optimal_idx = results_df['total_cost'].idxmin()
optimal_threshold = results_df.loc[optimal_idx, 'threshold']
optimal_cost = results_df.loc[optimal_idx, 'total_cost']

print(f"Optimal threshold: {optimal_threshold:.2f}")
print(f"Minimum total cost: £{optimal_cost:,.0f}")

In [ ]:
# =============================================================================
# Visualize cost vs threshold
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Total cost vs threshold
ax1 = axes[0]
ax1.plot(results_df['threshold'], results_df['total_cost']/1e6, 'b-', linewidth=2)
ax1.axvline(x=0.5, color='red', linestyle='--', label='Default (0.5)')
ax1.axvline(x=optimal_threshold, color='green', linestyle='--', label=f'Optimal ({optimal_threshold:.2f})')
ax1.scatter([optimal_threshold], [optimal_cost/1e6], color='green', s=100, zorder=5)
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Total Cost (£ millions)')
ax1.set_title('Total Cost vs Classification Threshold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right plot: FP and FN vs threshold
ax2 = axes[1]
ax2.plot(results_df['threshold'], results_df['FP'], 'b-', linewidth=2, label='False Positives')
ax2.plot(results_df['threshold'], results_df['FN'], 'r-', linewidth=2, label='False Negatives')
ax2.axvline(x=optimal_threshold, color='green', linestyle='--', label=f'Optimal ({optimal_threshold:.2f})')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Count')
ax2.set_title('Error Counts vs Threshold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key insight: Lower threshold catches more defaults (fewer FN)")
print("   but also denies more good borrowers (more FP)")

In [ ]:
# =============================================================================
# Compare metrics at default vs optimal threshold
# =============================================================================

# Calculate results at both thresholds
result_default = calculate_cost_at_threshold(y_test, y_prob_gb, 0.5, COST_FP, COST_FN)
result_optimal = calculate_cost_at_threshold(y_test, y_prob_gb, optimal_threshold, COST_FP, COST_FN)

comparison = pd.DataFrame([
    {'Threshold': 0.50, **result_default},
    {'Threshold': optimal_threshold, **result_optimal}
])[['Threshold', 'FP', 'FN', 'total_cost', 'precision', 'recall']]

comparison.columns = ['Threshold', 'False Positives', 'False Negatives', 'Total Cost (£)', 'Precision', 'Recall']

print("Comparison: Default vs Optimal Threshold")
print("="*70)
print(comparison.to_string(index=False))

cost_saved = result_default['total_cost'] - result_optimal['total_cost']
pct_saved = cost_saved / result_default['total_cost'] * 100 if result_default['total_cost'] > 0 else 0

print(f"\n💰 Cost saved by using optimal threshold: £{cost_saved:,.0f} ({pct_saved:.1f}% reduction)")

### 🤖 AI Prompt Exercise 2: Explain Threshold Selection

**Task:** Ask an AI to help explain the threshold optimization to a business stakeholder.

**Copy this prompt:**

```
I need to explain threshold selection to my manager. Here's the situation:

- Default threshold (0.5): We catch [X]% of defaults but have £[X] million in total costs
- Optimal threshold ([YOUR VALUE]): We catch [X]% of defaults but deny more good borrowers
- Cost of denying a good borrower: £500
- Cost of approving a defaulter: £5,000

Write a brief explanation (4-5 sentences) of:
1. Why we shouldn't use 0.5 as the threshold
2. Why the optimal threshold is lower
3. The business trade-off we're making
```

**Write the key points:**

*Your notes:*



---

## Part 6: From Models to Business Decisions

Now let's use our model to make actual lending decisions.

In [ ]:
# =============================================================================
# Create a decision framework based on predicted probabilities
# =============================================================================

# We can create multiple decision tiers based on risk:
#
# Tier 1: Very low risk → Approve at standard rate
# Tier 2: Moderate risk → Approve at higher rate
# Tier 3: High risk → Deny (or require collateral)

def make_lending_decision(prob_default, threshold_deny=0.20, threshold_premium=0.10):
    """
    Make lending decision based on predicted default probability.
    
    Parameters:
    -----------
    prob_default : float, predicted probability of default
    threshold_deny : float, above this → deny the loan
    threshold_premium : float, above this but below deny → approve at premium rate
    
    Returns:
    --------
    str : decision (Approve Standard, Approve Premium, or Deny)
    """
    if prob_default >= threshold_deny:
        return 'Deny'
    elif prob_default >= threshold_premium:
        return 'Approve (Premium Rate)'
    else:
        return 'Approve (Standard Rate)'

In [ ]:
# =============================================================================
# Apply decisions to test set
# =============================================================================

# Create a results DataFrame
decisions_df = pd.DataFrame({
    'actual_default': y_test.values,
    'prob_default': y_prob_gb,
    'decision': [make_lending_decision(p) for p in y_prob_gb]
})

# Summarize decisions
decision_summary = decisions_df.groupby('decision').agg({
    'actual_default': ['count', 'sum', 'mean']
}).round(3)

decision_summary.columns = ['Count', 'Actual Defaults', 'Default Rate']

print("Lending Decisions Summary:")
print("="*60)
print(decision_summary)
print(f"\nTotal loans in test set: {len(decisions_df):,}")

In [ ]:
# =============================================================================
# Calculate expected portfolio loss
# =============================================================================

# For approved loans, we can estimate expected losses:
# Expected Loss = Probability of Default × Loan Amount × Loss Given Default (LGD)
#
# LGD is typically 40-60% (we don't lose everything in a default)

LGD = 0.5  # Assume we recover 50% of the loan in case of default

# Get approved loans (both standard and premium)
approved_mask = decisions_df['decision'].isin(['Approve (Standard Rate)', 'Approve (Premium Rate)'])
approved_probs = y_prob_gb[approved_mask]

# Get loan amounts for approved loans (if loan_amnt column exists)
if 'loan_amnt' in X_test.columns:
    approved_amounts = X_test.loc[approved_mask.values, 'loan_amnt'].values
    
    # Calculate expected loss for each approved loan
    expected_losses = approved_probs * approved_amounts * LGD
    
    total_exposure = approved_amounts.sum()
    total_expected_loss = expected_losses.sum()
    
    print("Portfolio Risk Summary (Approved Loans):")
    print("="*50)
    print(f"Number of approved loans: {approved_mask.sum():,}")
    print(f"Total loan exposure: £{total_exposure:,.0f}")
    print(f"Expected loss (PD × LGD): £{total_expected_loss:,.0f}")
    print(f"Expected loss rate: {total_expected_loss/total_exposure:.2%}")
else:
    print("Portfolio Risk Summary (Approved Loans):")
    print("="*50)
    print(f"Number of approved loans: {approved_mask.sum():,}")
    print(f"Average predicted default probability: {approved_probs.mean():.2%}")

### 🤖 AI Prompt Exercise 3: Build a Risk-Based Pricing Function

**Task:** Ask an AI to help you build a complete risk-based pricing function.

**Copy this prompt:**

```
Write a Python function called `calculate_interest_rate` that:

1. Takes predicted probability of default as input
2. Returns an appropriate interest rate based on risk tier:
   - P(default) < 0.05: Base rate (8%)
   - 0.05 <= P(default) < 0.10: Base rate + 2%
   - 0.10 <= P(default) < 0.15: Base rate + 5%
   - 0.15 <= P(default) < 0.20: Base rate + 8%
   - P(default) >= 0.20: Return None (deny loan)

Include docstring and example usage.
```

**Paste and run:**

In [ ]:
# Paste AI-generated code here:


---

## Part 7: Key Takeaways

### Summary

In [ ]:
# =============================================================================
# Final model comparison
# =============================================================================

# Calculate AUC for all models
final_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'AUC': [
        roc_auc_score(y_test, y_prob_logreg),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_gb)
    ]
}).sort_values('AUC', ascending=False)

print("Final Model Comparison:")
print("="*40)
print(final_comparison.to_string(index=False))

### Key Lessons

1. **Accuracy is misleading** with imbalanced data — use precision, recall, F1, AUC

2. **AUC measures ranking ability** — useful for comparing models before choosing threshold

3. **Threshold of 0.5 is rarely optimal** — choose based on costs of different errors

4. **Costs are asymmetric** in credit risk — missing a defaulter costs much more than denying a good borrower

5. **Models enable decisions**:
   - Approve/deny at different thresholds
   - Risk-based pricing (higher probability → higher rate)
   - Portfolio expected loss calculation

### Practical Considerations

| Aspect | Logistic Regression | Tree Ensembles (RF, Gradient Boosting) |
|--------|---------------------|-----------------------------|
| Accuracy | Lower | Higher |
| Interpretability | High (coefficients) | Low (black box) |
| Regulatory acceptance | Easy | Harder |
| Calibration | Usually good | May need adjustment |